# 第7章：建立聊天應用程式
## OpenAI API 快速入門

本筆記本改編自包含存取 [Azure OpenAI](notebook-azure-openai.ipynb) 服務筆記本的 [Azure OpenAI 範例資源庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)。

Python OpenAI API 也可支援 Azure OpenAI 模型，僅需少許修改。詳細了解差異請參閱此處：[如何使用 Python 在 OpenAI 與 Azure OpenAI 端點之間切換](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# 概觀  
「大型語言模型是將文字映射到文字的函數。給定一個輸入的文字字串，大型語言模型嘗試預測接下來會出現的文字」(1)。這份「快速入門」筆記本將介紹使用者大型語言模型的高階概念、開始使用 AML 所需的核心套件、對提示設計的簡單介紹，以及幾個不同使用案例的短範例。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立您的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 憑證](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文字](#1.-summarize-text)  
[2. 分類文字](#2.-classify-text)  
[3. 產生新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考文獻](#references)


### 建立你的第一個提示  
這個簡短的練習將提供如何向OpenAI模型提交提示以執行單純任務「摘要」的基本介紹。


<strong>步驟</strong>:  
1. 在你的 Python 環境中安裝 OpenAI 函式庫  
2. 載入標準輔助函式庫並設置你為所建立的 OpenAI 服務設定的典型 OpenAI 安全憑證  
3. 選擇用於任務的模型  
4. 為模型建立一個簡單提示  
5. 將請求提交至模型 API！


### 1. 安裝 OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. 匯入輔助函式庫並實例化憑證


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. 尋找合適的模型  
像 GPT-4o 和 GPT-4o mini 這樣的模型能夠理解和產生自然語言，並且在 OpenAI 平台上提供不同的運算能力與速度，適合不同的任務需求。


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. 提示設計  

「大型語言模型的魔力在於，透過對海量文本進行預測誤差最小化的訓練，模型最終學會了對這些預測有用的概念。例如，它們學會了像」(1)：  

* 如何拼寫  
* 語法如何運作  
* 如何改寫同義句  
* 如何回答問題  
* 如何進行對話  
* 如何用多種語言寫作  
* 如何寫程式碼  
* 等等。  

#### 如何控制大型語言模型  
「在所有輸入大型語言模型的輸入中，最具影響力的遠遠是文字提示(1)。  

大型語言模型可以通過幾種方式來提示以產生輸出：  

指示：告訴模型你想要什麼  
補全：促使模型完成你想要的開頭內容  
示範：向模型展示你想要的，方式為：  
在提示中提供幾個範例  
在微調訓練數據集中提供數百或數千個範例」  



#### 建立提示有三個基本指導原則：  

<strong>示範與說明</strong>。清楚說明你想要什麼，無論是透過指示、範例或兩者結合。如果你想讓模型將列表項目按字母順序排列，或根據情緒對段落進行分類，就明確地告訴它這是你想要的。  

<strong>提供高質量的資料</strong>。如果你試圖建立分類器或讓模型遵循某種模式，請確保有足夠的範例。務必校對範例——模型通常足夠聰明，可看透基本拼寫錯誤並給出答覆，但它也可能假設這是刻意為之，並影響回應結果。  

**檢查你的設定。** temperature 和 top_p 設定決定模型生成回應時的確定性。如果你要求的回應只有一個正確答案，則應將這些設定調低；如果你希望得到更多元的回應，則可將它們調高。人們在這些設定上最常犯的錯誤是以為它們是「智慧度」或「創造力」的控制開關。  


來源：https://learn.microsoft.com/azure/ai-foundry/openai/overview  


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### 重複相同的呼叫，結果會如何比較？


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## 總結文本  
#### 挑戰  
透過在文本段落末尾加入「tl;dr:」來總結文本。請注意模型如何在沒有額外指令的情況下理解並執行多項任務。你可以嘗試比tl;dr更具描述性的提示，以調整模型的行為並自訂你所獲得的摘要(3)。  

近期的研究顯示，透過在大量文本語料庫上進行預訓練，接著在特定任務上微調，可以在許多自然語言處理任務和基準測試中獲得顯著提升。雖然此方法在架構上通常不依賴任務，但仍需數千或數萬個任務特定的微調數據集。相比之下，人類通常只需少量範例或簡單指令就能執行新的語言任務——這是目前的自然語言處理系統仍普遍難以做到的。在此，我們展示擴大語言模型規模能大幅提升任務無關的少量範例性能，有時甚至能與先前最先進的微調方法競爭。 



總結  


# 多個使用案例的練習  
1. 摘要文字  
2. 分類文字  
3. 產生新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 分類文本  
#### 挑戰  
在推理時將項目分類到提供的類別中。在以下範例中，我們在提示中同時提供類別和要分類的文本(*playground_reference)。 

客戶詢問：您好，我筆記型電腦鍵盤上的一個按鍵最近壞了，我需要更換一個新的：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 產生新產品名稱
#### 挑戰
從範例詞彙中創建產品名稱。這裡我們在提示中包含了我們將要為其產生名稱的產品信息。我們也提供了一個類似範例來展示我們想要的模式。我們同時將溫度值設定較高，以增加隨機性和創新回應。

產品描述：家庭用奶昔製造機
種子詞：快速，健康，緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：可適合任何腳型的鞋子。
種子詞：適應性、合腳、全合腳。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微調 GPT 模型以分類文本的最佳實踐](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 如需更多幫助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
